## Step 1 — Install dependencies

In [1]:
!pip install -q kaggle scikit-image scikit-learn opencv-python-headless

## Step 2 — Upload your Kaggle API token


In [ ]:
from google.colab import files

uploaded = files.upload()  # select kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data
!unzip -q /content/data/microsoft-catsvsdogs-dataset.zip -d /content/data
!ls /content/data

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import random

random.seed(42)
np.random.seed(42)

In [ ]:
DATA_DIR = '/content/data/PetImages'
CATEGORIES = ['Cat', 'Dog']
IMG_SIZE = 64
SAMPLES_PER_CLASS = 2000   # set to None to use the full ~12,500 images per class (slower)

def load_and_extract_features(data_dir, categories, img_size, samples_per_class=None):
    features, labels = [], []
    for label, category in enumerate(categories):
        folder = os.path.join(data_dir, category)
        filenames = os.listdir(folder)
        random.shuffle(filenames)
        if samples_per_class is not None:
            filenames = filenames[:samples_per_class]

        loaded = 0
        for fname in filenames:
            fpath = os.path.join(folder, fname)
            try:
                img = cv2.imread(fpath)
                if img is None:
                    continue  # corrupted / unreadable file
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                img = cv2.resize(img, (img_size, img_size))

                feat = hog(
                    img,
                    orientations=9,
                    pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2),
                    block_norm='L2-Hys'
                )
                features.append(feat)
                labels.append(label)
                loaded += 1
            except Exception:
                continue  # skip any bad/corrupted image
        print(f"{category}: loaded {loaded} valid images")

    return np.array(features), np.array(labels)

X, y = load_and_extract_features(DATA_DIR, CATEGORIES, IMG_SIZE, SAMPLES_PER_CLASS)
print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

In [ ]:
svm_model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_model.fit(X_train_scaled, y_train)
print("Training complete.")

In [ ]:
run_grid_search = False  # set True to enable

if run_grid_search:
    param_grid = {
        'C': [1, 10, 100],
        'gamma': ['scale', 0.01, 0.001],
        'kernel': ['rbf', 'linear']
    }
    grid = GridSearchCV(SVC(random_state=42), param_grid, cv=3, verbose=2, n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    print("Best params:", grid.best_params_)
    svm_model = grid.best_estimator_

In [ ]:
y_pred = svm_model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - SVM (Cats vs Dogs)')
plt.show()

In [ ]:
def show_predictions(data_dir, categories, model, scaler, img_size, n=6):
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    picks = []
    for category in categories:
        folder = os.path.join(data_dir, category)
        files_ = os.listdir(folder)
        random.shuffle(files_)
        for f in files_[:n // 2]:
            picks.append((os.path.join(folder, f), category))
    random.shuffle(picks)

    for ax, (fpath, true_label) in zip(axes, picks):
        img = cv2.imread(fpath)
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (img_size, img_size))
        feat = hog(resized, orientations=9, pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2), block_norm='L2-Hys')
        feat_scaled = scaler.transform([feat])
        pred = categories[model.predict(feat_scaled)[0]]

        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        color = 'green' if pred == true_label else 'red'
        ax.set_title(f"True: {true_label}\nPred: {pred}", color=color)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_predictions(DATA_DIR, CATEGORIES, svm_model, scaler, IMG_SIZE, n=6)